In [0]:
# Run in a Databricks notebook cell to install required libraries
# OR add via Cluster → Libraries → Install New

%pip install faker==24.0.0
%pip install confluent-kafka==2.3.0
%pip install boto3==1.34.0

# Restart Python after pip installs
%restart_python

In [0]:
# setup/01_catalog_schema_setup.py
# CATALOG & SCHEMA SETUP (Unity Catalog)

def setup_catalog_and_schemas(spark):
    """
    Create the catalog and schema structure for the fraud detection platform 
    using Unity Catalog's 3-tier namespace.
    """
    catalog_name = "fintech_fraud"
    
    # Catalog creation
    spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog_name}")
    print(f"✅ Initialized Unity Catalog: {catalog_name}")

    # Schema (Database) creation
    schemas = {
        "bronze": "Raw ingested data — Bronze layer",
        "silver": "Cleaned and CDC-processed data — Silver layer",
        "gold":   "Fraud analytics aggregates — Gold layer",
    }

    for schema_name, comment in schemas.items():
        # In Unity Catalog, managed tables automatically route to the storage 
        # root defined at the Catalog or Metastore level. No LOCATION required.
        spark.sql(f"""
            CREATE SCHEMA IF NOT EXISTS {catalog_name}.{schema_name}
            COMMENT '{comment}'
        """)
        print(f"✅ Created schema: {catalog_name}.{schema_name}")

    # Verify
    print("\n📋 Schemas in catalog:")
    spark.sql(f"SHOW SCHEMAS IN {catalog_name}").show(truncate=False)


# PIPELINE CONFIGURATION CONSTANTS

class FraudPipelineConfig:
    """
    Centralized configuration for business thresholds and rules.
    (Table namespaces and Kafka endpoints are managed dynamically by DLT).
    """
    
    # Catalog configuration 
    CATALOG_NAME        = "fintech_fraud"

    # Fraud detection thresholds 
    HIGH_VALUE_THRESHOLD     = 10000.0   # USD
    VELOCITY_WINDOW_MINUTES  = 60        # transactions per hour
    VELOCITY_COUNT_LIMIT     = 15        # max txns/hour before flagging
    IMPOSSIBLE_TRAVEL_KMH    = 900       # km/h (faster than plane = suspicious)
    HIGH_RISK_MERCHANT_SCORE = 0.7       # 0-1 scale
    AML_STRUCTURING_AMOUNT   = 9000.0    # Just below $10K reporting threshold

    # Data quality thresholds
    MAX_NULL_RATE_PCT        = 5.0       # % nulls allowed before quarantine
    MIN_AMOUNT               = 0.01
    MAX_AMOUNT               = 1_000_000.0


# Print config summary and execute schema setup
if __name__ == "__main__":
    # 1. Print the configuration
    config = FraudPipelineConfig()
    print("=" * 60)
    print("  FINTECH FRAUD DETECTION — UC PIPELINE CONFIGURATION")
    print("=" * 60)
    print(f"  Catalog Name:     {config.CATALOG_NAME}")
    print(f"  High Value Limit: ${config.HIGH_VALUE_THRESHOLD:,.0f}")
    print(f"  Velocity Limit:   {config.VELOCITY_COUNT_LIMIT} txns/{config.VELOCITY_WINDOW_MINUTES}min")
    print("=" * 60)
    
    # 2. Actually trigger the schema creation!
    print("\nStarting Catalog and Schema Setup...")
    try:
        # Databricks automatically provides the 'spark' session variable globally
        setup_catalog_and_schemas(spark)
    except NameError:
        print("⚠️ 'spark' session not found. This script must be executed within a Databricks environment.")